# Retail Data Warehouse - Star Schema
## STEP 1: Create Database and Schema

In [ ]:
%%sql -r setup_db
CREATE DATABASE IF NOT EXISTS RETAIL_DW;
USE DATABASE RETAIL_DW;
CREATE SCHEMA IF NOT EXISTS STAR_SCHEMA;
USE SCHEMA STAR_SCHEMA;

## STEP 2: Create Internal Stage for CSV files

In [ ]:
%%sql -r create_stage
CREATE STAGE IF NOT EXISTS RETAIL_STAGE FILE_FORMAT = (
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    FIELD_DELIMITER = ','
);

## STEP 3: Upload CSV files to stage (via Snowsight UI or PUT command)

## STEP 4: Create Dimension Tables

In [ ]:
%%sql -r create_dim_customer
-- DIM_CUSTOMER: Stores customer information
CREATE OR REPLACE TABLE DIM_CUSTOMER (
    CUSTOMER_ID INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    CITY VARCHAR(50),
    STATE VARCHAR(50),
    MEMBERSHIP VARCHAR(20)
);

In [ ]:
%%sql -r create_dim_product
-- DIM_PRODUCT: Stores product information
CREATE OR REPLACE TABLE DIM_PRODUCT (
    PRODUCT_ID INT PRIMARY KEY,
    PRODUCT_NAME VARCHAR(100),
    CATEGORY VARCHAR(50),
    BRAND VARCHAR(50),
    PRICE DECIMAL(10,2)
);

In [ ]:
%%sql -r create_dim_branch
-- DIM_BRANCH: Stores branch information
CREATE OR REPLACE TABLE DIM_BRANCH (
    BRANCH_ID INT PRIMARY KEY,
    BRANCH_NAME VARCHAR(100),
    CITY VARCHAR(50),
    STATE VARCHAR(50),
    REGION VARCHAR(20),
    MANAGER_NAME VARCHAR(100)
);

In [ ]:
%%sql -r create_dim_date
-- DIM_DATE: Stores date/calendar information
CREATE OR REPLACE TABLE DIM_DATE (
    DATE_ID INT PRIMARY KEY,
    DATE DATE,
    DAY INT,
    DAY_NAME VARCHAR(20),
    WEEK_NO INT,
    MONTH VARCHAR(20),
    QUARTER VARCHAR(5),
    YEAR INT,
    IS_WEEKEND VARCHAR(5)
);

## STEP 5: Create Fact Table

In [ ]:
%%sql -r create_fact_sales
-- FACT_SALES: Central fact table storing all sales transactions
CREATE OR REPLACE TABLE FACT_SALES (
    SALE_ID INT PRIMARY KEY,
    CUSTOMER_ID INT,
    PRODUCT_ID INT,
    BRANCH_ID INT,
    DATE_ID INT,
    QUANTITY INT,
    TOTAL_AMOUNT DECIMAL(12,2),
    FOREIGN KEY (CUSTOMER_ID) REFERENCES DIM_CUSTOMER(CUSTOMER_ID),
    FOREIGN KEY (PRODUCT_ID) REFERENCES DIM_PRODUCT(PRODUCT_ID),
    FOREIGN KEY (BRANCH_ID) REFERENCES DIM_BRANCH(BRANCH_ID),
    FOREIGN KEY (DATE_ID) REFERENCES DIM_DATE(DATE_ID)
);

## STEP 6: Load Data into Dimension Tables

In [ ]:
%%sql -r load_customers
COPY INTO DIM_CUSTOMER
FROM @RETAIL_STAGE/customers.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_DELIMITER = ',' FIELD_OPTIONALLY_ENCLOSED_BY = '"');

In [ ]:
%%sql -r load_products
COPY INTO DIM_PRODUCT
FROM @RETAIL_STAGE/products.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_DELIMITER = ',' FIELD_OPTIONALLY_ENCLOSED_BY = '"');

In [ ]:
%%sql -r load_branches
COPY INTO DIM_BRANCH
FROM @RETAIL_STAGE/branches.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_DELIMITER = ',' FIELD_OPTIONALLY_ENCLOSED_BY = '"');

In [ ]:
%%sql -r load_dates
COPY INTO DIM_DATE
FROM @RETAIL_STAGE/calendar.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_DELIMITER = ',' FIELD_OPTIONALLY_ENCLOSED_BY = '"');

## STEP 7: Load Data into Fact Table

In [ ]:
%%sql -r load_sales
COPY INTO FACT_SALES
FROM @RETAIL_STAGE/sales.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_DELIMITER = ',' FIELD_OPTIONALLY_ENCLOSED_BY = '"');

## STEP 8: Verify Data Load

In [ ]:
%%sql -r verify_load
SELECT 'DIM_CUSTOMER' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM DIM_CUSTOMER
UNION ALL
SELECT 'DIM_PRODUCT', COUNT(*) FROM DIM_PRODUCT
UNION ALL
SELECT 'DIM_BRANCH', COUNT(*) FROM DIM_BRANCH
UNION ALL
SELECT 'DIM_DATE', COUNT(*) FROM DIM_DATE
UNION ALL
SELECT 'FACT_SALES', COUNT(*) FROM FACT_SALES;

## STEP 9: Analytical Queries (Business Reports)

In [ ]:
%%sql -r report1_customer_sales
-- Report 1: Customer-wise Sales Report
SELECT c.CUSTOMER_NAME, c.CITY, c.STATE, c.MEMBERSHIP,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.CUSTOMER_ID = c.CUSTOMER_ID
GROUP BY c.CUSTOMER_NAME, c.CITY, c.STATE, c.MEMBERSHIP
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report2_product_revenue
-- Report 2: Product-wise Revenue Report
SELECT p.PRODUCT_NAME, p.CATEGORY, p.BRAND, p.PRICE,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.PRODUCT_ID = p.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY, p.BRAND, p.PRICE
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report3_branch_revenue
-- Report 3: Branch-wise Revenue Report
SELECT b.BRANCH_NAME, b.CITY, b.STATE, b.REGION, b.MANAGER_NAME,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.BRANCH_ID = b.BRANCH_ID
GROUP BY b.BRANCH_NAME, b.CITY, b.STATE, b.REGION, b.MANAGER_NAME
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report4_state_revenue
-- Report 4: State-wise Revenue Report
SELECT b.STATE, b.REGION,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.BRANCH_ID = b.BRANCH_ID
GROUP BY b.STATE, b.REGION
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report5_monthly_revenue
-- Report 5: Monthly Revenue Report
SELECT d.MONTH, d.YEAR,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_DATE d ON f.DATE_ID = d.DATE_ID
GROUP BY d.MONTH, d.YEAR
ORDER BY d.YEAR, d.MONTH;

In [ ]:
%%sql -r report6_quarterly_revenue
-- Report 6: Quarterly Revenue Report
SELECT d.QUARTER, d.YEAR,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_DATE d ON f.DATE_ID = d.DATE_ID
GROUP BY d.QUARTER, d.YEAR
ORDER BY d.YEAR, d.QUARTER;

In [ ]:
%%sql -r report7_top10_customers
-- Report 7: Top 10 Customers
SELECT c.CUSTOMER_NAME, c.CITY, c.STATE,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.CUSTOMER_ID = c.CUSTOMER_ID
GROUP BY c.CUSTOMER_NAME, c.CITY, c.STATE
ORDER BY TOTAL_REVENUE DESC
LIMIT 10;

In [ ]:
%%sql -r report8_top10_products
-- Report 8: Top 10 Products
SELECT p.PRODUCT_NAME, p.CATEGORY, p.BRAND,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.PRODUCT_ID = p.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY, p.BRAND
ORDER BY TOTAL_REVENUE DESC
LIMIT 10;

In [ ]:
%%sql -r report9_top10_branches
-- Report 9: Top 10 Performing Branches
SELECT b.BRANCH_NAME, b.CITY, b.REGION,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.BRANCH_ID = b.BRANCH_ID
GROUP BY b.BRANCH_NAME, b.CITY, b.REGION
ORDER BY TOTAL_REVENUE DESC
LIMIT 10;

In [ ]:
%%sql -r report10_category_revenue
-- Report 10: Category-wise Revenue
SELECT p.CATEGORY,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.PRODUCT_ID = p.PRODUCT_ID
GROUP BY p.CATEGORY
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report11_customer_trend
-- Report 11: Customer Purchase Trend
SELECT c.CUSTOMER_NAME, d.MONTH, d.YEAR,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.CUSTOMER_ID = c.CUSTOMER_ID
JOIN DIM_DATE d ON f.DATE_ID = d.DATE_ID
GROUP BY c.CUSTOMER_NAME, d.MONTH, d.YEAR
ORDER BY c.CUSTOMER_NAME, d.YEAR, d.MONTH;

In [ ]:
%%sql -r report12_product_dashboard
-- Report 12: Product Performance Dashboard
SELECT p.PRODUCT_NAME, p.CATEGORY, p.BRAND, p.PRICE,
    COUNT(f.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE,
    AVG(f.TOTAL_AMOUNT) AS AVG_REVENUE_PER_TRANSACTION
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.PRODUCT_ID = p.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY, p.BRAND, p.PRICE
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report13_branch_dashboard
-- Report 13: Branch Performance Dashboard
SELECT b.BRANCH_NAME, b.CITY, b.STATE, b.REGION, b.MANAGER_NAME,
    COUNT(f.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE,
    AVG(f.TOTAL_AMOUNT) AS AVG_REVENUE_PER_TRANSACTION
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.BRANCH_ID = b.BRANCH_ID
GROUP BY b.BRANCH_NAME, b.CITY, b.STATE, b.REGION, b.MANAGER_NAME
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report14_regional_analysis
-- Report 14: Regional Sales Analysis
SELECT b.REGION,
    COUNT(DISTINCT b.BRANCH_ID) AS NUM_BRANCHES,
    COUNT(f.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.BRANCH_ID = b.BRANCH_ID
GROUP BY b.REGION
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r report15_daily_trend
-- Report 15: Sales Trend Analysis (Day-wise)
SELECT d.DATE, d.DAY_NAME, d.IS_WEEKEND,
    COUNT(f.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(f.QUANTITY) AS TOTAL_QUANTITY,
    SUM(f.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM FACT_SALES f
JOIN DIM_DATE d ON f.DATE_ID = d.DATE_ID
GROUP BY d.DATE, d.DAY_NAME, d.IS_WEEKEND
ORDER BY d.DATE;